# Caso 3 · Cox07 — Magnitudes positivas y presupuesto
Edición docente adaptada de [A. M. Mayoral, GLM, caso 3](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html). [Revisión docente del 25-11-2026](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso3).

Seis unidades, explicaciones propias y código R sobre la cartera del equipo 07. No es una reproducción literal de los apuntes ni un informe final. El ejemplo industrial se traslada a seguros: máquina → póliza, avería → parte, coste de avería → coste por siniestro, gasto máquina-año → coste póliza-año. No inventamos equivalencias de covariables. No se presupone que los modelos funcionen mejor por aparecer en el temario.

## 0 · Preparación
El RDS conserva los parámetros del DGP; los CSV son las tres tablas oficiales con las huellas de la versión utilizada en el caso 2. La unidad de exposición es el año. Separamos ajuste y reserva por póliza. Sexo no entra en la tarifa; tampoco variables que solo se conocen después del periodo predicho.

In [ ]:
SEMILLA_CURSO <- 20252026L
set.seed(SEMILLA_CURSO)
options(repr.plot.width=11, repr.plot.height=6)
paquetes <- c("dplyr","ggplot2","readr","glmmTMB","DHARMa","MASS","car","statmod")
faltan <- setdiff(paquetes, rownames(installed.packages()))
if(length(faltan)) install.packages(faltan, repos="https://cloud.r-project.org")
stopifnot(all(vapply(paquetes, requireNamespace, logical(1), quietly=TRUE)))
library(dplyr)
library(ggplot2)
theme_set(theme_minimal(base_size=14))
base_equipo <- "https://raw.githubusercontent.com/asunmayoral/GLM/b05aeaecde9dca0049ea0ca55fbe162a9b964cbc/caso2/datos_equipos_auto/equipo_07/"
huellas <- c("cartera.csv"="6a8a1cd5a22ec0813be10b4d2173b28f","panel.csv"="813f20b73d15d78bb5777a21268d07c8","siniestros.csv"="9523e2bd2665c492d8dc1fbe5502d7de","cartera.rds"="949486c1f1c5b51ef43f441748e1932d")
archivo_equipo <- function(nombre) {
  ruta <- file.path("datos", nombre)
  if(!file.exists(ruta)) {
    ruta <- file.path(tempdir(), nombre)
    if(!file.exists(ruta)) download.file(paste0(base_equipo,nombre),ruta,mode="wb",quiet=TRUE)
  }
  stopifnot(unname(tools::md5sum(ruta)) == huellas[[nombre]])
  ruta
}
cartera_original <- readr::read_csv(archivo_equipo("cartera.csv"),show_col_types=FALSE)
panel <- readr::read_csv(archivo_equipo("panel.csv"),show_col_types=FALSE)
siniestros <- readr::read_csv(archivo_equipo("siniestros.csv"),show_col_types=FALSE)
cartera_rds <- readRDS(archivo_equipo("cartera.rds"))
stopifnot(!anyDuplicated(cartera_original$id_poliza),sum(cartera_original$n_siniestros)==nrow(siniestros),sum(panel$n_partes_anual)==nrow(siniestros))
set.seed(SEMILLA_CURSO)
ids_ajuste <- sample(cartera_original$id_poliza,floor(.75*nrow(cartera_original)))
ids_reserva <- setdiff(cartera_original$id_poliza,ids_ajuste)
print(c(polizas=nrow(cartera_original),anualidades=nrow(panel),partes=nrow(siniestros)))

## 0 · Construir las unidades de análisis
La exposición se mide en años con expo_anual. El coste de un parte está en euros. Los predictores son características conocidas al inicio; excluimos coste, conteos futuros y bonus-malus actual. Los costes cero se conservan en el panel, no en Gamma. La reserva se separa por póliza para evitar que un mismo cliente aparezca en ajuste y validación.

In [ ]:
rasgos <- c("id_poliza","id_agencia","id_region","edad_conductor","antiguedad_vehiculo","tipo_vehiculo","zona_circulacion","uso","km_anuales","sexo")
clientes <- cartera_original |> select(all_of(rasgos)) |> mutate(across(c(id_agencia,id_region,tipo_vehiculo,zona_circulacion,uso,sexo),factor), edad10=edad_conductor/10, veh10=antiguedad_vehiculo/10, km10=km_anuales/10000)
stopifnot(all(is.finite(siniestros$coste_k)),all(siniestros$coste_k>=0),!anyDuplicated(panel[c("id_poliza","anio")]))
partes <- left_join(siniestros,clientes,by="id_poliza")
anual <- panel |> left_join(siniestros |> group_by(id_poliza,anio) |> summarise(coste=sum(coste_k),.groups="drop"),by=c("id_poliza","anio")) |> mutate(coste=coalesce(coste,0)) |> left_join(clientes,by="id_poliza")
stopifnot(abs(sum(anual$coste)-sum(partes$coste_k))<.01,all(anual$expo_anual>=0),all(anual$coste[anual$expo_anual==0]==0))
vars <- c("edad10","veh10","tipo_vehiculo","zona_circulacion","uso","km10")
stopifnot(all(complete.cases(clientes[vars])))
positivos <- filter(partes,coste_k>0)
print(c(partes=nrow(partes),ceros=sum(partes$coste_k==0),polizas=nrow(clientes),anualidades=nrow(anual)))
A <- filter(positivos,id_poliza %in% ids_ajuste)
V <- filter(positivos,id_poliza %in% ids_reserva)
P <- filter(anual,expo_anual>0,id_poliza %in% ids_ajuste)
Q <- filter(anual,expo_anual>0,id_poliza %in% ids_reserva)
f <- reformulate(vars,"coste_k")
g <- reformulate(vars,"coste")
ggplot(A,aes(coste_k))+geom_histogram(bins=50,fill="gold",colour="white")+labs(x="Coste por parte (€)",y="Partes")
ggplot(A,aes(tipo_vehiculo,coste_k))+geom_violin(fill="gold",alpha=.6)+geom_boxplot(width=.12)+scale_y_log10()+labs(y="Coste (€), escala log")

## 3.1 · GLM Gamma
[Unidad original](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html#sec-u31)

### 1.1 Contexto: qué modela
Un parte positivo tiene coste continuo, asimétrico y no negativo. La Gamma excluye el cero. La normal puede asignar probabilidad a costes negativos. El coste de una póliza sin partes se estudia en la unidad 3.3, no se elimina de la cartera.

In [ ]:
print(summary(A$coste_k))
print(c(CV=sd(A$coste_k)/mean(A$coste_k),cola_95=quantile(A$coste_k,.95)))
mOLS <- lm(f,data=A)
plot(fitted(mOLS),residuals(mOLS),pch=16,col="#77777755",xlab="Media ajustada (€)",ylab="Residuo (€)")
abline(h=0,col="tomato")
print(sum(predict(mOLS,V)<0))
print(A |> group_by(tipo_vehiculo) |> summarise(n=n(),media=mean(coste_k),sd=sd(coste_k),CV=sd/media,.groups="drop"))

### 1.2 Gamma en la familia exponencial
Para forma α y escala μ/α, E[Y]=μ y Var(Y)=μ²/α. En la familia exponencial podemos escribir θ=−1/μ, b(θ)=−log(−θ), φ=1/α. Entonces b′(θ)=μ y b″(θ)=μ². El enlace canónico es el inverso negativo; R admite inverse (signo opuesto), y log es útil para interpretar multiplicadores. El coeficiente de variación es constante si φ es constante.

In [ ]:
mu0 <- mean(A$coste_k)
rejG <- expand.grid(y=seq(mu0/100,mu0*6,length.out=400),alpha=c(.5,1,2,5))
rejG$densidad <- dgamma(rejG$y,shape=rejG$alpha,scale=mu0/rejG$alpha)
ggplot(rejG,aes(y,densidad,colour=factor(alpha)))+geom_line(linewidth=1)+labs(x="Coste (€)",colour="Forma α")
ggplot(A,aes(coste_k))+stat_ecdf(geom="step",colour="goldenrod")+scale_x_log10()+labs(x="Coste (€), escala log",y="Proporción acumulada")

### 1.3 ¿De qué depende el coste?
Exploramos tipo de vehículo, zona, edad y kilometraje. La interacción pregunta si un efecto depende de otro predictor. El gráfico no prueba causalidad; tampoco garantiza que la interacción se sostenga fuera de muestra.

In [ ]:
ggplot(A,aes(tipo_vehiculo,coste_k,fill=zona_circulacion))+geom_boxplot(outlier.alpha=.15)+scale_y_log10()
ggplot(A,aes(veh10*10,coste_k,colour=tipo_vehiculo))+geom_point(alpha=.15)+geom_smooth(method="loess",se=FALSE)+scale_y_log10()+labs(x="Antigüedad vehículo (años)")
print(A |> group_by(tipo_vehiculo,zona_circulacion) |> summarise(n=n(),media=mean(coste_k),.groups="drop"))

### 1.4 Modelización y estimación
Con enlace log, η=Xβ y μ=exp(η). La log-verosimilitud Gamma produce el score Xᵀ(y/μ−1)/φ. Para el enlace log, los pesos de Fisher scoring son constantes (1/φ); las respuestas de trabajo sí cambian en cada iteración. El enlace inverso requiere predicciones admisibles.

In [ ]:
m0 <- glm(f,data=A,family=gaussian())
mG <- glm(f,data=A,family=Gamma("log"))
mI <- try(glm(f,data=A,family=Gamma("inverse"),start=c(1/mean(A$coste_k),rep(0,length(coef(mG))-1))),silent=TRUE)
summary(mG)
print(exp(cbind(razon=coef(mG),confint.default(mG))))
drop1(mG,test="F")
AIC(m0,mG)
if(inherits(mI,"glm") && mI$converged) print(AIC(mG,mI))
diagnostico <- function(m) {
  plot(fitted(m),residuals(m,type="pearson"),xlab="Media ajustada",ylab="Residuo Pearson",pch=16,col="#77777755")
  abline(h=0,col="tomato")
}
diagnostico(mG)
pr <- predict(mG,newdata=V,type="link",se.fit=TRUE)
head(data.frame(media=exp(pr$fit),inf=exp(pr$fit-1.96*pr$se.fit),sup=exp(pr$fit+1.96*pr$se.fit)))

In [ ]:
Xg <- model.matrix(mG)
yg <- A$coste_k
bg <- c(log(mean(yg)),rep(0,ncol(Xg)-1))
# Fisher scoring con reducción del paso si empeora la devianza.
devg <- function(b){mu<-exp(drop(Xg%*%b));2*sum(yg/mu-1-log(yg/mu))}
for(i in 1:100){
  paso<-drop(qr.solve(Xg,yg/exp(drop(Xg%*%bg))-1))
  escala<-1
  while(escala>1e-8 && devg(bg+escala*paso)>devg(bg)) escala<-escala/2
  bg<-bg+escala*paso
  if(max(abs(escala*paso))<1e-8) break
}
print(cbind(IWLS=bg,glm=coef(mG)))

### 1.5 Interpretación
exp(β) multiplica el coste medio por parte. La dispersión φ mide la variabilidad relativa; CV=√φ. No es una tasa de frecuencia. Los intervalos Wald se calculan primero en escala η y después se transforman a euros.

In [ ]:
print(exp(cbind(multiplicador=coef(mG),confint.default(mG))))
print(c(phi=summary(mG)$dispersion,CV=sqrt(summary(mG)$dispersion)))

### 1.6 Bondad de ajuste y selección
Contraste F de tipo II, residuos DHARMa, curvatura y ajuste de la cola. Comparamos términos cuadráticos y spline natural sobre la antigüedad del vehículo. Elegimos con evidencia de ajuste, no con la reserva. La especificación lineal queda como referencia común para comparar las familias de la unidad siguiente.

In [ ]:
print(car::Anova(mG,type=2,test.statistic="F"))
curva <- update(mG,.~.+I(veh10^2))
spline <- update(mG,.~.-veh10+splines::ns(veh10,df=4))
inter <- update(mG,.~.+tipo_vehiculo:zona_circulacion)
print(anova(mG,curva,test="F"))
print(anova(mG,spline,test="F"))
print(anova(mG,inter,test="F"))
print(AIC(mG,curva,spline,inter))
simG<-DHARMa::simulateResiduals(mG,n=250)
plot(simG)
print(DHARMa::testDispersion(simG))
print(DHARMa::testQuantiles(simG,predictor=A$veh10,plot=FALSE))

### 1.7 Modelo y predicción
Cinco perfiles oficiales del caso 2. El intervalo de confianza estima la media del coste por parte. El intervalo Gamma de predicción incluye el ruido de un parte individual, condicionado a los parámetros estimados.

In [ ]:
phiG <- summary(mG)$dispersion
perfiles <- data.frame(perfil=c("A Joven urbano","B Familiar rural","C Moto urbana","D Furgoneta comercial","E Conductor mayor"),edad10=c(23,45,30,40,70)/10,veh10=c(8,6,4,10,15)/10,km10=c(15000,12000,8000,35000,6000)/10000,tipo_vehiculo=c("turismo","turismo","moto","furgoneta","turismo"),zona_circulacion=c("urbana","rural","urbana","mixta","mixta"),uso=c("particular","particular","particular","comercial","particular"),expo_anual=1)
for(v in c("tipo_vehiculo","zona_circulacion","uso")) perfiles[[v]] <- factor(perfiles[[v]],levels=levels(clientes[[v]]))
prG <- predict(mG,perfiles,type="link",se.fit=TRUE)
print(data.frame(perfil=perfiles$perfil,media=exp(prG$fit),ICinf=exp(prG$fit-1.96*prG$se.fit),ICsup=exp(prG$fit+1.96*prG$se.fit),IPinf=qgamma(.025,shape=1/phiG,scale=exp(prG$fit)*phiG),IPsup=qgamma(.975,shape=1/phiG,scale=exp(prG$fit)*phiG)))

## 3.2 · Elegir familia y escala
[Unidad original](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html#sec-u32)

### 2.1 Qué distingue a una familia de otra
La familia normal tiene varianza constante, Gamma tiene potencia 2 e inversa gaussiana potencia 3. Agrupamos por media ajustada para ver si la dispersión observada se parece a alguna ley. La nube es descriptiva, no una prueba de identificación de la familia.

In [ ]:
nube <- A |> mutate(mu=fitted(mG),grupo=ntile(mu,10)) |> group_by(grupo) |> summarise(media=mean(mu),varianza=mean((coste_k-mu)^2),.groups="drop")
ggplot(nube,aes(media,varianza))+geom_point(size=3)+geom_smooth(method="lm",se=FALSE)+scale_x_log10()+scale_y_log10()+labs(x="Media ajustada",y="Varianza residual")

### 2.2 Inversa gaussiana
La varianza aumenta más deprisa que en Gamma. Ajustamos el mismo predictor y examinamos inferencia, residuos y predicción. No damos por supuesto que la cola sea mejor.

In [ ]:
mIG <- glm(f,data=A,family=inverse.gaussian("log"))
print(summary(mIG))
print(car::Anova(mIG,type=2,test.statistic="F"))
plot(DHARMa::simulateResiduals(mIG,n=250))
prIG<-predict(mIG,perfiles,type="link",se.fit=TRUE)
phiIG<-summary(mIG)$dispersion
print(data.frame(perfil=perfiles$perfil,media=exp(prIG$fit),ICinf=exp(prIG$fit-1.96*prIG$se.fit),ICsup=exp(prIG$fit+1.96*prIG$se.fit),IPinf=statmod::qinvgauss(.025,mean=exp(prIG$fit),dispersion=phiIG),IPsup=statmod::qinvgauss(.975,mean=exp(prIG$fit),dispersion=phiIG)))

### 2.3 Lognormal y log-OLS
E[exp(Z)] no es exp(E[Z]). Si el error logarítmico es normal con varianza constante, el factor corrector es exp(σ²/2). Duan estima el factor mediante la media de exp(residuo), sin exigir normalidad. Bootstrap agrupado por póliza incorpora incertidumbre del ajuste y del factor. La transformación del intervalo de predicción logarítmico no requiere smearing.

In [ ]:
mLN <- lm(update(f,log(coste_k)~.),data=A)
duan <- mean(exp(residuals(mLN)))
print(c(Duan=duan,normal=exp(sigma(mLN)^2/2)))
print(data.frame(perfil=perfiles$perfil,mediana=exp(predict(mLN,perfiles)),media=exp(predict(mLN,perfiles))*duan,exp(predict(mLN,perfiles,interval="prediction"))))
plot(mLN,which=1:2)

In [ ]:
set.seed(SEMILLA_CURSO)
grupos <- split(A,A$id_poliza)
boot_duan <- replicate(200,{
  d <- do.call(rbind,sample(grupos,length(grupos),replace=TRUE))
  fit <- lm(update(f,log(coste_k)~.),data=d)
  exp(predict(fit,perfiles))*mean(exp(residuals(fit)))
})
print(t(apply(boot_duan,1,quantile,c(.025,.975))))

### 2.4 Elegir, comparar y conectar
Para comparar AIC en la misma escala, sumamos 2Σlog(y) al AIC de log-OLS (jacobiano). Comparamos errores y calibración en euros fuera de muestra. El subconjunto de costes observados altos sirve para examinar errores en la cola, no como prueba aislada de calibración.

In [ ]:
metricas <- function(y,p) c(MAE=mean(abs(y-p)),RMSE=sqrt(mean((y-p)^2)),sesgo=mean(p-y),observado=sum(y),predicho=sum(p))
print(c(Gamma=AIC(mG),IG=AIC(mIG),Lognormal=AIC(mLN)+2*sum(log(A$coste_k))))
preds <- list(Gamma=predict(mG,V,type="response"),IG=predict(mIG,V,type="response"),Lognormal=exp(predict(mLN,V))*duan)
print(sapply(preds,function(p) metricas(V$coste_k,p)))
umbral<-quantile(A$coste_k,.95)
cola<-V$coste_k>=umbral
if(any(cola)) print(sapply(preds,function(p) metricas(V$coste_k[cola],p[cola])))

## 3.3 · El coste agregado: la familia Tweedie
[Unidad original](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html#sec-u33)

### 3.1 El hueco en la recta de las varianzas
El coste anual es la suma de costes de partes. Puede ser exactamente cero o continuo positivo. Gamma no admite el cero. La familia Tweedie con 1<p<2 lo admite mediante una mezcla Poisson–Gamma. No confundimos ceros de coste con ceros estructurales.

In [ ]:
print(c(proporcion_cero=mean(P$coste==0),coste_total=sum(P$coste),prima_anual=sum(P$coste)/sum(P$expo_anual)))
ggplot(P,aes(coste))+geom_histogram(bins=50,fill="gold",colour="white")

### 3.2 Tweedie en acción
Var(S)=φμ^p. Seguimos el script docente con offset log(exposición). Contrastamos también tasa anual con pesos de exposición: ambas especificaciones comparten media, pero no varianza. Para el modelo con offset, el p y φ estimados describen la distribución de coste total bajo esa especificación.

In [ ]:
P$tasa <- P$coste/P$expo_anual
Q$tasa <- Q$coste/Q$expo_anual
ft <- reformulate(c(vars,"offset(log(expo_anual))"),"coste")
mT <- glmmTMB::glmmTMB(ft,data=P,family=glmmTMB::tweedie(link="log"))
mT0 <- glmmTMB::glmmTMB(coste~offset(log(expo_anual)),data=P,family=glmmTMB::tweedie(link="log"))
mTasa <- glmmTMB::glmmTMB(reformulate(vars,"tasa"),data=P,weights=expo_anual,family=glmmTMB::tweedie(link="log"))
valido <- function(m) inherits(m,"glmmTMB") && isTRUE(m$sdr$pdHess) && isTRUE(m$fit$convergence==0)
print(c(p=glmmTMB::family_params(mT)[[1]],phi=sigma(mT)))
print(c(convergencia=valido(mT),tasa=valido(mTasa)))
print(anova(mT0,mT))
pT<-glmmTMB::family_params(mT)[[1]]
phiT<-sigma(mT)
muT<-predict(mT,type="response")
lambdaT<-muT^(2-pT)/(phiT*(2-pT))
print(c(ceros_observados=sum(P$coste==0),ceros_esperados=sum(exp(-lambdaT))))
if(valido(mT)) plot(DHARMa::simulateResiduals(mT,n=250))

### 3.3 Los tres caminos
A: frecuencia × severidad. B: probabilidad de coste positivo × coste medio positivo. C: Tweedie directa. La frecuencia de referencia es NB2 sobre póliza-año, no una importación exacta del modelo aprobado en C2. Si hay partes de coste cero, multiplicamos la severidad positiva por la proporción de partes positivos estimada en ajuste; esto supone que esa proporción no varía con el perfil. Los tres caminos predicen euros por periodo; para un año completo equivalen a euros por póliza-año.

In [ ]:
fn <- reformulate(c(vars,"offset(log(expo_anual))"),"n_partes_anual")
mF <- MASS::glm.nb(fn,data=P)
mHay <- glm(reformulate(c(vars,"offset(log(expo_anual))"),"I(coste > 0)"),data=P,family=binomial("cloglog"))
mPos <- glm(reformulate(c(vars,"log(expo_anual)"),"coste"),data=filter(P,coste>0),family=Gamma("log"))
probpos <- mean(partes$coste_k[partes$id_poliza %in% ids_ajuste]>0)
tres_caminos <- function(d) data.frame(FrecSev=predict(mF,d,type="response")*predict(mG,d,type="response")*probpos,DosPartes=predict(mHay,d,type="response")*predict(mPos,d,type="response"),Tweedie=predict(mT,d,type="response"))
predQ <- tres_caminos(Q)
print(sapply(predQ,function(p) metricas(Q$coste,p)))
print(cbind(perfiles["perfil"],tres_caminos(perfiles)))

cal <- Q |> mutate(pred=predQ$Tweedie,grupo=ntile(pred/expo_anual,10)) |> group_by(grupo) |> summarise(obs=sum(coste)/sum(expo_anual),pred=sum(pred)/sum(expo_anual),.groups="drop")
ggplot(cal,aes(pred,obs))+geom_abline(slope=1,intercept=0)+geom_point(colour="tomato",size=3)
print(metricas(Q$coste,predict(mTasa,Q,type="response")*Q$expo_anual))

In [ ]:
comparar<-cbind(Q[c("id_poliza","expo_anual","coste")],predQ)
print(colSums(predQ))
print(sum(Q$coste))
ggplot(comparar,aes(FrecSev/expo_anual,Tweedie/expo_anual))+geom_point(alpha=.15)+geom_abline(slope=1,intercept=0,colour="tomato")+labs(x="Frecuencia × severidad (€/año)",y="Tweedie (€/año)")
ggplot(comparar,aes(DosPartes/expo_anual,Tweedie/expo_anual))+geom_point(alpha=.15)+geom_abline(slope=1,intercept=0,colour="tomato")+labs(x="Dos partes (€/año)",y="Tweedie (€/año)")

## 3.4 · Efectos aleatorios y credibilidad
[Unidad original](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html#sec-u34)

### 4.1–4.2 Contexto y modelo mixto
Dos siniestros de una misma póliza pueden compartir características no medidas. Un intercepto aleatorio en escala log permite una severidad sistemáticamente mayor o menor. También exploramos agencia. No atribuimos causalidad al efecto estimado.

In [ ]:
g0 <- glmmTMB::glmmTMB(f,data=A,family=Gamma("log"))
g1 <- glmmTMB::glmmTMB(update(f,.~.+(1|id_poliza)),data=A,family=Gamma("log"))
g2 <- glmmTMB::glmmTMB(update(f,.~.+(1|id_poliza)+(1|id_agencia)),data=A,family=Gamma("log"))
print(c(independiente=valido(g0),poliza=valido(g1),agencia=valido(g2)))
print(AIC(g0,g1,g2))
print(glmmTMB::VarCorr(g2))

### 4.3 Interpretación
Si u~N(0,σ²), exp(u) es el multiplicador de riesgo. Fijar u=0 produce la mediana entre pólizas de un mismo perfil; integrar da exp(η+σ²/2). Los efectos estimados se encogen hacia cero cuando hay poca información. Gamma de glmmTMB usa sigma² como dispersión.

In [ ]:
if(valido(g1)){
  u<-glmmTMB::ranef(g1)$cond$id_poliza
  cr<-data.frame(id_poliza=as.integer(rownames(u)),u=u[,1]) |> left_join(A |> count(id_poliza),by="id_poliza")
  print(ggplot(cr,aes(n,u))+geom_point(alpha=.3)+geom_hline(yintercept=0)+labs(x="Partes observados",y="Efecto aleatorio"))
  v<-glmmTMB::VarCorr(g1)$cond$id_poliza[1,1]
  print(c(phi=sigma(g1)^2,sd=sqrt(v),multiplicador_marginal=exp(v/2)))
}

### 4.4 Bondad, diagnóstico y selección
Un modelo con pendiente aleatoria por agencia permite que la antigüedad del vehículo tenga efectos diferentes. Es una hipótesis, no una estructura obligatoria. Revisamos convergencia, Hessiano y diagnóstico. El contraste de varianza cero está en la frontera del espacio de parámetros.

In [ ]:
g3<-try(glmmTMB::glmmTMB(update(f,.~.+(1|id_poliza)+(1+veh10|id_agencia)),data=A,family=Gamma("log")),silent=TRUE)
if(valido(g3)) print(AIC(g2,g3)) else message("Pendiente aleatoria no validada")
if(valido(g2)) plot(DHARMa::simulateResiduals(g2,n=250))

### 4.5 Historial en el coste anual: bonus-malus
El nivel anterior al último año puede explicar el coste de ese año. El nivel posterior no puede hacerlo sin fuga. Excluimos pólizas de un solo año. Estimamos el efecto adicional al perfil con referencia nivel 3. La corrección individual mediante GLMM es una alternativa a bonus-malus; multiplicar ambas sin comprobación puede penalizar dos veces. Utilizamos Tweedie como ejemplo provisional, no como ganador confirmado.

In [ ]:
valido <- function(m) inherits(m,"glmmTMB") && isTRUE(m$sdr$pdHess) && isTRUE(m$fit$convergence==0)
bm <- cartera_original |> select(id_poliza,bonus_malus_prev,bonus_malus_act,baja) |> mutate(across(c(bonus_malus_prev,bonus_malus_act),~factor(as.character(.x),levels=as.character(1:5))))
ultimo <- anual |> group_by(id_poliza) |> filter(anio==max(anio)) |> ungroup() |> left_join(bm,by="id_poliza")
# El año máximo debe ser mayor que uno y tener exposición positiva.
ultimo <- filter(ultimo,anio>1,expo_anual>0)
ultimo$bonus_malus_prev <- relevel(droplevels(ultimo$bonus_malus_prev),ref="3")
mBM0 <- glmmTMB::glmmTMB(ft,data=ultimo,family=glmmTMB::tweedie(link="log"))
mBM <- glmmTMB::glmmTMB(update(ft,.~.+bonus_malus_prev),data=ultimo,family=glmmTMB::tweedie(link="log"))
print(AIC(mBM0,mBM))
print(exp(glmmTMB::fixef(mBM)$cond))
print(exp(confint(mBM,parm="beta_",method="wald")))
hist <- filter(anual,expo_anual>0)
mHist <- glmmTMB::glmmTMB(update(ft,.~.+(1|id_poliza)),data=hist,family=glmmTMB::tweedie(link="log"))
print(glmmTMB::VarCorr(mHist))
if(valido(mHist)) {
  efectos <- glmmTMB::ranef(mHist)$cond$id_poliza
  cred <- data.frame(id_poliza=as.integer(rownames(efectos)),correccion=exp(efectos[,1])) |> left_join(bm,by="id_poliza") |> left_join(hist |> group_by(id_poliza) |> summarise(expo=sum(expo_anual),.groups="drop"),by="id_poliza")
  print(head(cred))
  print(ggplot(cred,aes(bonus_malus_act,correccion))+geom_violin(fill="gold")+geom_boxplot(width=.12)+geom_hline(yintercept=1))
  print(ggplot(cred,aes(expo,log(correccion)))+geom_point(alpha=.3)+geom_hline(yintercept=0))
  plot(DHARMa::simulateResiduals(mHist,n=250))
}

### 4.6 Modelo y predicción
Comparamos pólizas conocidas y nuevas. El BLUP usa datos de la póliza conocida y no debe usarse en una validación de años que contribuyeron a estimarlo. Para una póliza nueva integramos los efectos aleatorios. El gráfico de credibilidad anterior relaciona historial y corrección; no representa una prueba causal.

In [ ]:
if(valido(mHist)){
  vh<-glmmTMB::VarCorr(mHist)$cond$id_poliza[1,1]
  print(c(sd=sqrt(vh),factor_marginal=exp(vh/2)))
  print(head(cred |> arrange(desc(correccion))))
}

## 3.5 · Del modelo al presupuesto
[Unidad original](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html#sec-u35)

### 5.1–5.2 Escenario del año próximo
Cartera viva (baja=0), un año completo por póliza, edades del vehículo y conductor aumentadas un año. Los nuevos clientes tipo entran en nivel 3. No proyectamos inflación, bajas futuras ni crecimiento no pedido en el caso 3. La media del presupuesto no es el gasto que necesariamente se producirá.

In [ ]:
viva <- clientes |> left_join(bm,by="id_poliza") |> filter(baja==0) |> mutate(edad10=edad10+.1,veh10=veh10+.1,expo_anual=1)
niveles <- levels(ultimo$bonus_malus_prev)
viva$bonus_malus_prev <- factor(as.character(viva$bonus_malus_act),levels=niveles)
stopifnot(!anyNA(viva$bonus_malus_prev))
perfiles$bonus_malus_prev <- factor("3",levels=niveles)
print(table(viva$bonus_malus_prev))

### 5.3 Media del presupuesto e incertidumbre de estimación
B=Σμ_i. Con enlace log, el gradiente de B respecto a β es Σμ_i x_i. El método delta aproxima Var(B)=gᵀ Cov(β) g. No se suman los errores estándar de cada póliza, porque comparten coeficientes. El escenario usa el modelo con bonus-malus ajustado en últimos años válidos.

In [ ]:
stopifnot(valido(mBM))
mu<-predict(mBM,viva,type="response")
b<-glmmTMB::fixef(mBM)$cond
X<-model.matrix(reformulate(c(vars,"bonus_malus_prev")),viva)
X<-X[,names(b),drop=FALSE]
stopifnot(max(abs(exp(drop(X%*%b))-mu))<1e-5*max(mu))
Vb<-vcov(mBM)$cond
grad<-colSums(X*mu)
se<-sqrt(drop(t(grad)%*%Vb%*%grad))
print(c(media=sum(mu),ICinf=max(0,sum(mu)-1.96*se),ICsup=sum(mu)+1.96*se))
print(data.frame(perfil=perfiles$perfil,prima=predict(mBM,perfiles,type="response")))

### 5.4 Intervalo predictivo
La representación compuesta de Tweedie tiene N~Poisson(λ), costes Gamma de forma α y escala τ; λ=μ^(2−p)/(φ(2−p)), α=(2−p)/(p−1), τ=φ(p−1)μ^(p−1). Si N=0 el coste es cero. Simulamos el ruido del proceso y la incertidumbre de β. No se incluye incertidumbre de φ y p ni dependencia residual entre pólizas. Por tanto los márgenes son condicionales al modelo.

In [ ]:
p<-glmmTMB::family_params(mBM)[[1]]
phi<-sigma(mBM)
stopifnot(p>1,p<2,phi>0)
rtweedie_pg<-function(mu,p,phi){
  N<-rpois(length(mu),mu^(2-p)/(phi*(2-p)))
  salida<-numeric(length(mu))
  pos<-N>0
  salida[pos]<-rgamma(sum(pos),shape=N[pos]*(2-p)/(p-1),scale=phi*(p-1)*mu[pos]^(p-1))
  salida
}
set.seed(SEMILLA_CURSO)
B<-MASS::mvrnorm(1000,b,Vb)
azar<-replicate(1000,sum(rtweedie_pg(mu,p,phi)))
total<-apply(B,1,function(beta)sum(rtweedie_pg(exp(drop(X%*%beta)),p,phi)))
print(rbind(proceso=quantile(azar,c(.05,.5,.9,.95)),total=quantile(total,c(.05,.5,.9,.95))))
ggplot(data.frame(presupuesto=total),aes(presupuesto))+geom_histogram(bins=45,fill="gold",colour="white")+geom_vline(xintercept=quantile(total,.9),colour="tomato")

### 5.5 Validación hacia atrás
Retiramos el último periodo de pólizas con historial. Reajustamos un GLMM sin esos costes y predecimos el último periodo. No usamos el bonus-malus actual de esos clientes. No es una prueba de calendario común: las pólizas entran en fechas distintas.

In [ ]:
hist <- hist |> group_by(id_poliza) |> mutate(ultimo_anio=anio==max(anio),varios=n()>1) |> ungroup()
train_h <- filter(hist,!ultimo_anio | !varios)
test_h <- filter(hist,ultimo_anio,varios)
mBack <- glmmTMB::glmmTMB(update(ft,.~.+(1|id_poliza)),data=train_h,family=glmmTMB::tweedie(link="log"))
if(valido(mBack)) print(metricas(test_h$coste,predict(mBack,test_h,type="response",allow.new.levels=TRUE)))
print(metricas(test_h$coste,predict(mBack,test_h,type="response",re.form=NA,allow.new.levels=TRUE)))

### 5.6 Escenarios de renovación
El equivalente al intervalo de mantenimiento industrial es la corrección de tarifa por historial. Comparamos la tarifa de referencia con las correcciones de bonus-malus y credibilidad como alternativas, no acumuladas. El efecto sobre la recaudación no implica un cambio causal del riesgo ni de la siniestralidad.

In [ ]:
sin_bm<-predict(mBM,transform(viva,bonus_malus_prev=factor("3",levels=niveles)),type="response")
print(c(referencia_nivel3=sum(sin_bm),tarifa_bonus_malus=sum(mu)))
if(valido(mHist)){
  cond<-predict(mHist,viva,type="response",allow.new.levels=TRUE)
  fija<-predict(mHist,viva,type="response",re.form=NA,allow.new.levels=TRUE)
  v<-glmmTMB::VarCorr(mHist)$cond$id_poliza[1,1]
  print(c(historial_condicional=sum(cond),efecto_cero=sum(fija),marginal_nuevos=sum(fija*exp(v/2))))
}

### 5.7 Implicaciones para dirección financiera
Presentar el presupuesto medio, el intervalo de estimación, el intervalo predictivo y el percentil 90. La diferencia entre el percentil 90 y la media cuantifica un margen de proceso bajo el modelo; no es una exigencia de capital regulatorio. Separar prima pura de gastos, impuestos y margen comercial.

In [ ]:
print(c(prima_media=sum(mu)/nrow(viva),presupuesto_medio=sum(mu),p90=quantile(total,.9),margen_p90=quantile(total,.9)-sum(mu)))

## 3.6 · Estudio de caso: tarifa completa y presupuesto
[Unidad original](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html#sec-u36)

### 6.1–6.3 Datos, encargo y hoja de ruta
El comité necesita cinco respuestas: coste por parte y factores; familia y escala; comparación de los tres caminos; bonus-malus e historial; presupuesto de la cartera viva e intervalos para los cinco perfiles. Las preguntas son obligatorias y forman una historia común.

### 6.4 Artículo de investigación
Introducción, datos y métodos, resultados, discusión, conclusiones y reproducibilidad. Justificar el tratamiento de ceros, exposición, familia, enlace, efectos aleatorios y validación. No se requiere validar contra el DGP en la versión docente actual; el RDS se conserva por trazabilidad.

[Scripts originales de las seis unidades y maestro](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso3/scripts). Esta edición cubre los apartados principales, no cada bloque ni el texto literal. Pendiente ejecutar en R, revisar convergencia y contrastar la tarifa de frecuencia con el modelo final del equipo. Los ejemplos con Tweedie en P4/P5 son provisionales.

In [ ]:
sessionInfo()